# Russian ASR ITN Hybrid — Inverse Text Normalization

Обратная текстовая нормализация (ITN) для русского языка в контексте ASR:
преобразование числительных, записанных словами, в цифровую форму.

```
двести пятьдесят рублей  →  250 рублей
первого числа            →  1 числа
две тысячи двадцать третий → 2023
```

Проект реализует **два параллельных подхода**:

| Подход | Метод | Точность |
|--------|-------|----------|
| Rule-based парсер | Словари + mag-based state machine | **99.8%** на calibration.f |
| ruT5-small + LoRA | Seq2Seq (65M params, LoRA r=8) | **57.3%** на clean synthetic |

Оба подхода объединены в гибрид: парсер по умолчанию, T5 при низкой уверенности.

In [1]:
import sys, os, re, inspect, collections, json, random
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
import pandas as pd
import pyarrow.feather as feather
import numpy as np

---
## 1. Датасеты

### 1.1 calibration.f (500 строк)

Эталонный датасет. Короткие ASR-фразы (средняя длина ~30 символов) с числовыми
контекстами: цены, бюджеты, сроки, проценты. Минимум пунктуации, чистый русский язык.

### 1.2 synthetic.f (16 618 строк)

Сгенерирован через `scripts/generate_synthetic.py`. Содержит два подмножества:

| Подмножество | Строк | Доля |
|---|---|---|
| clean | 4 890 | 29.4% |
| noisy (ASR-шум) | 11 728 | 70.6% |

**4 источника:** template (45.1%), grouping (36.1%), ordinal (18.1%), news (0.7%)

### 1.3 real.f (172 строки)

Реальные тексты из Wikipedia API и новостных RSS. Длинные контексты (средняя
длина 123 символа), смешанный русско-английский текст, пунктуация, дефисы.

| Характеристика | calibration.f | synthetic.f | real.f |
|---|---|---|---|
| Длина текста (ср.) | ~30 | 26.5 | 122.9 |
| Пунктуация | Почти нет | Минимум | Скобки, %, :, дефисы |
| Язык | Русский | Русский | Русский + английский |
| ASR-шум | Нет | 70.6% noisy | Нет |

In [2]:
cal = feather.read_table("data/calibration.f").to_pandas()
syn = feather.read_table("data/synthetic.f").to_pandas()
real = feather.read_table("data/real.f").to_pandas()

print(f"calibration.f: {len(cal)} rows")
print(f"  task len: mean={cal['task_text'].str.len().mean():.1f}")
print(f"synthetic.f:  {len(syn)} rows")
print(f"  noise_level: {dict(syn['noise_level'].value_counts())}")
print(f"  source: {dict(syn['source'].value_counts())}")
print(f"  num_type: {dict(syn['num_type'].value_counts())}")
print(f"real.f:       {len(real)} rows")
print(f"  task len: mean={real['task_text'].str.len().mean():.1f}")

print("\n--- calibration.f samples ---")
for _, r in cal.head(3).iterrows():
    print(f"  TASK: {r['task_text'][:70]}\n  GT:   {r['ground_truth'][:70]}\n")

calibration.f: 500 rows
  task len: mean=230.9
synthetic.f:  16618 rows
  noise_level: {'noisy': 11728, 'clean': 4890}
  source: {'template': 7500, 'grouping': 6000, 'ordinal': 3000, 'news': 118}
  num_type: {'cardinal': 13618, 'ordinal': 3000}
real.f:       172 rows
  task len: mean=135.4

--- calibration.f samples ---
  TASK: вам нужно будет перейти в раздел настройки личного кабинета и внести в
  GT:   вам нужно будет перейти в раздел настройки личного кабинета и внести в

  TASK: нажмите на свободную область экрана и вы увидите в нижней части вот та
  GT:   нажмите на свободную область экрана и вы увидите в нижней части вот та

  TASK: пятьсот восемьдесят рублей если брать среднее значение то это примерно
  GT:   580 рублей если брать среднее значение то это примерно стоимость упако



---
## 2. Rule-based парсер

### 2.1 Архитектура

```
Входной текст
    │
    ▼
_asr_preprocess() — regex-коррекция ASR-ошибок (мягкий знак, "двеси")
    │
    ▼
Токенизация по пробелам
    │
    ▼
┌──────────────────────────────────────┐
│       Два параллельных пути          │
│                                      │
│  Path A (legacy, default):           │
│    normalize_text()                  │
│    → lookup_word()                   │
│    → parse_number_group() (mag)      │
│                                      │
│  Path B (sequence, --parser-type):   │
│    normalize_text_sequence()          │
│    → classify() (token_classifier)   │
│    → parse_sequence() (state machine)│
└──────────────────────────────────────┘
    │
    ▼
Результат: числа цифрами + остальной текст
```

### 2.2 История улучшений (legacy)

**97.6% → 99.8%** (488/500 → 499/500). Исправлено 11 из 12 ошибок:

| Фаза | Что делали | Поймано |
|---|---|---|
| 1 | Словарные фиксы (тыща, тысячам, ё-формы) | 5 |
| 2 | Логические фиксы (тыщ контекст, fused, mag=0 repeat) | 3 |
| 3 | Два multiplier одного ранга → enum | 1 |
| 4 | ASR regex-препроцессинг ("двеси" сдвиг разрядов) | 1 |
| 5 | Levenshtein fallback для ordinals | пограничные |
| 6 | Root-based динамическое расширение словаря | — |
| 7 | pymorphy2-дисамбигуация (сто/три/сорок) | — |

**Оставшаяся ошибка:** `дваста` → GT сохраняет ASR-ошибку, parser прав для ITN.

### 2.3 Словари числительных

Словари содержат все возможные словоформы: падежи, числа, ASR-искажения.
Формат: `слово → (числовое_значение, mag)`.

**mag (порядок величины):**
- 0 = единицы (0-9) и собирательные
- 1 = числа 10-19
- 2 = десятки 20-90
- 3 = сотни 100-900
- 4 = тысячи
- 5 = миллионы
- 6 = миллиарды

In [3]:
UNITS = {
    "ноль": (0, 0), "один": (1, 0), "одна": (1, 0), "одно": (1, 0),
    "одного": (1, 0), "одной": (1, 0), "одному": (1, 0),
    "одним": (1, 0), "одном": (1, 0), "одни": (1, 0), "одних": (1, 0),
    "два": (2, 0), "две": (2, 0), "двух": (2, 0), "двум": (2, 0), "двумя": (2, 0),
    "три": (3, 0), "трёх": (3, 0), "трех": (3, 0), "трем": (3, 0), "тремя": (3, 0),
    "четыре": (4, 0), "четырёх": (4, 0), "четырех": (4, 0),
    "четырем": (4, 0), "четырьмя": (4, 0),
    "пять": (5, 0), "пяти": (5, 0), "пятью": (5, 0),
    "шесть": (6, 0), "шести": (6, 0), "шестью": (6, 0),
    "семь": (7, 0), "семи": (7, 0), "семью": (7, 0),
    "восемь": (8, 0), "восьми": (8, 0), "восемью": (8, 0),
    "девять": (9, 0), "девяти": (9, 0), "девятью": (9, 0),
}
COLLECTIVES = {"двое": (2, 0), "трое": (3, 0), "четверо": (4, 0),
    "пятеро": (5, 0), "шестеро": (6, 0), "семеро": (7, 0),
    "восьмеро": (8, 0), "девятеро": (9, 0), "десятеро": (10, 0)}
TEENS = {
    "десять": (10, 1), "десяти": (10, 1), "десятью": (10, 1),
    "одиннадцать": (11, 1), "одиннадцати": (11, 1),
    "двенадцать": (12, 1), "двенадцати": (12, 1),
    "тринадцать": (13, 1), "тринадцати": (13, 1),
    "четырнадцать": (14, 1), "четырнадцати": (14, 1),
    "пятнадцать": (15, 1), "пятнадцати": (15, 1),
    "шестнадцать": (16, 1), "шестнадцати": (16, 1),
    "семнадцать": (17, 1), "семнадцати": (17, 1),
    "восемнадцать": (18, 1), "восемнадцати": (18, 1),
    "девятнадцать": (19, 1), "девятнадцати": (19, 1),
}
TENS = {
    "двадцать": (20, 2), "двадцати": (20, 2),
    "тридцать": (30, 2), "тридцати": (30, 2), "тридцатью": (30, 2),
    "сорок": (40, 2), "сорока": (40, 2),
    "пятьдесят": (50, 2), "пятидесяти": (50, 2),
    "шестьдесят": (60, 2), "шестидесяти": (60, 2),
    "семьдесят": (70, 2), "семидесяти": (70, 2),
    "восемьдесят": (80, 2), "восьмидесяти": (80, 2),
    "девяносто": (90, 2), "девяноста": (90, 2),
}
HUNDREDS = {
    "сто": (100, 3), "ста": (100, 3),
    "двести": (200, 3), "двухсот": (200, 3), "двумстам": (200, 3),
    "двумястами": (200, 3), "двухстах": (200, 3),
    "двеси": (200, 3), "дваста": (200, 3),
    "триста": (300, 3), "трёхсот": (300, 3), "трехсот": (300, 3),
    "тремстам": (300, 3), "тремястами": (300, 3), "трёхстах": (300, 3),
    "четыреста": (400, 3), "четырёхсот": (400, 3), "четырехсот": (400, 3),
    "четыремстам": (400, 3), "четырьмястами": (400, 3),
    "пятьсот": (500, 3), "пятисот": (500, 3),
    "шестьсот": (600, 3), "шестисот": (600, 3),
    "семьсот": (700, 3), "семисот": (700, 3),
    "восемьсот": (800, 3), "восьмисот": (800, 3),
    "девятьсот": (900, 3), "девятисот": (900, 3),
}
THOUSANDS = {
    "тысяча": (1000, 4), "тысячи": (1000, 4), "тысяч": (1000, 4),
    "тысячу": (1000, 4), "тысяче": (1000, 4), "тысячей": (1000, 4),
    "тысячам": (1000, 4), "тысячами": (1000, 4), "тысячах": (1000, 4),
    "тыщ": (1000, 4), "тыщи": (1000, 4), "тыща": (1000, 4),
}
MILLIONS = {"миллион": (1000000, 5), "миллиона": (1000000, 5),
    "миллионов": (1000000, 5), "миллионом": (1000000, 5),
    "миллионам": (1000000, 5), "миллионами": (1000000, 5), "миллионах": (1000000, 5)}
BILLIONS = {"миллиард": (1000000000, 6), "миллиарда": (1000000000, 6),
    "миллиардов": (1000000000, 6), "миллиардам": (1000000000, 6),
    "миллиардами": (1000000000, 6), "миллиардах": (1000000000, 6)}

**ASR-ошибки** — словарь искажений -> каноническая форма.
Собран анализом calibration.f: ~40 вариантов покрывают >95% ASR-искажений.

In [4]:
ASR_ERRORS = {
    "двеси": "двести", "дваста": "двести",
    "тыщ": "тысяч", "тыщи": "тысячи",
    "петьсят": "пятьдесят", "петьдесят": "пятьдесят",
    "писят": "пятьдесят", "пядсят": "пятьдесят", "пьсят": "пятьдесят",
    "пядцить": "пятнадцать", "пятнадцить": "пятнадцать",
    "питнадцать": "пятнадцать", "петнадцать": "пятнадцать",
    "симнадцить": "семнадцать", "семнадцить": "семнадцать",
    "шиснатцать": "шестнадцать", "шеснатцать": "шестнадцать",
    "восимнадцить": "восемнадцать", "восемнадцить": "восемнадцать",
    "дивяносто": "девяносто", "дивяноста": "девяноста",
    "тысич": "тысяч", "тысеч": "тысяч", "тисяч": "тысяч", "тищ": "тысяч",
    "милион": "миллион", "милиона": "миллиона", "милионов": "миллионов",
    "тристи": "триста", "семсот": "семьсот",
    "восимсот": "восемьсот", "девятсот": "девятьсот",
    "двенацтать": "двенадцать", "тринацать": "тринадцать", "четырнацать": "четырнадцать",
    "восемсат": "восемьдесят", "восимдесат": "восемьдесят",
    "дисять": "десять", "дисят": "десять", "десят": "десять",
    "тыc": "тысяч", "тыс": "тысяч",
}

In [5]:
NUMERAL_DICT = {}
for d in (UNITS, TEENS, TENS, HUNDREDS, THOUSANDS, MILLIONS, BILLIONS):
    NUMERAL_DICT.update(d)
NUMERAL_DICT.update({"дветысячи": (2000, 0), "двесте": (200, 3), "двестипятьсот": (2500, 0)})
MULTIPLIERS = set()
for d in (THOUSANDS, MILLIONS, BILLIONS): MULTIPLIERS.update(d.keys())
print(f"NUMERAL_DICT: {len(NUMERAL_DICT)} entries, {len(MULTIPLIERS)} multipliers")
print(f"ASR_ERRORS: {len(ASR_ERRORS)} entries")

NUMERAL_DICT: 137 entries, 25 multipliers
ASR_ERRORS: 42 entries


**Порядковые числительные** — ~200 форм, включая ё-формы.

In [6]:
ORDINALS = {
    "первый": "1", "первого": "1", "первому": "1",
    "первым": "1", "первом": "1", "первая": "1", "первой": "1",
    "первую": "1", "первое": "1", "первые": "1", "первых": "1", "первыми": "1",
    "второй": "2", "второго": "2", "второму": "2",
    "вторым": "2", "втором": "2", "вторая": "2", "вторую": "2",
    "второе": "2", "вторые": "2", "вторых": "2", "вторыми": "2",
    "третий": "3", "третьего": "3", "третьему": "3",
    "третьим": "3", "третьем": "3", "третья": "3", "третьей": "3",
    "третью": "3", "третье": "3", "третьи": "3", "третьих": "3", "третьими": "3",
    "четвертый": "4", "четвертого": "4", "четвертому": "4",
    "четвертым": "4", "четвертом": "4",
    "четвертая": "4", "четвертой": "4", "четвертую": "4", "четвертое": "4",
    "четвертые": "4", "четвертых": "4",
    "пятый": "5", "пятого": "5", "пятому": "5",
    "пятым": "5", "пятом": "5",
    "пятая": "5", "пятой": "5", "пятую": "5", "пятое": "5",
    "пятые": "5", "пятых": "5",
    "шестой": "6", "шестого": "6", "шестому": "6",
    "шестым": "6", "шестом": "6",
    "шестая": "6", "шестую": "6", "шестое": "6",
    "шестые": "6", "шестых": "6",
    "седьмой": "7", "седьмого": "7", "седьмому": "7",
    "седьмым": "7", "седьмом": "7",
    "седьмая": "7", "седьмую": "7", "седьмое": "7",
    "седьмые": "7", "седьмых": "7",
    "восьмой": "8", "восьмого": "8", "восьмому": "8",
    "восьмым": "8", "восьмом": "8",
    "восьмая": "8", "восьмую": "8", "восьмое": "8",
    "восьмые": "8", "восьмых": "8",
    "девятый": "9", "девятого": "9", "девятому": "9",
    "девятым": "9", "девятом": "9",
    "девятая": "9", "девятой": "9", "девятую": "9",
    "девятое": "9", "девятые": "9", "девятых": "9",
    "десятый": "10", "десятого": "10",
    "одиннадцатый": "11", "одиннадцатого": "11", "одиннадцатое": "11",
    "двенадцатый": "12", "двенадцатого": "12", "двенадцатое": "12",
    "тринадцатый": "13", "тринадцатого": "13", "тринадцатое": "13",
    "четырнадцатый": "14", "четырнадцатое": "14",
    "пятнадцатый": "15", "пятнадцатое": "15",
    "шестнадцатый": "16", "шестнадцатое": "16",
    "семнадцатый": "17", "семнадцатое": "17",
    "восемнадцатый": "18", "восемнадцатое": "18",
    "девятнадцатый": "19", "девятнадцатое": "19",
    "двадцатый": "20", "тридцатый": "30",
    "сороковой": "40", "пятидесятый": "50",
    "шестидесятый": "60", "семидесятый": "70", "восьмидесятый": "80",
    "девяностый": "90",
    "сотый": "100", "двухсотый": "200", "двухсотая": "200", "двухсотую": "200",
    "трёхсотый": "300", "четырехсотый": "400",
    "пятисотый": "500", "шестисотый": "600",
    "семисотый": "700", "восьмисотый": "800", "девятисотый": "900",
}
ORDINAL_SET = set(ORDINALS.keys())
print(f"ORDINALS: {len(ORDINALS)} forms")

ORDINALS: 140 forms


---
## 3. Token Classifier (новый путь)

Единый классификатор, заменяющий ad-hoc `lookup_word` + `is_ordinal_word` +
`_cardinal_from_root` + `_fuzzy_ordinal_match` на систематический пайплайн.

**Пайплайн classify():**
1. Vague context "тыщ" -> пропуск (разговорное "примерно")
2. Exact dict match (O(1), быстрый путь)
3. Ordinal exact match
4. ASR error correction
5. Root-based regex (33 корня, сортировка по убыванию длины)
6. Fused compound split ("дветысячи" -> два+тысячи)

**TokenClass dataclass:**

| Поле | Тип | Описание |
|---|---|---|
| value | int | 5, 10, 100, 1000... |
| mag | int | 0=unit, 1=teen, 2=ten, 3=hundred, 4=thousand... |
| subtype | str | cardinal, ordinal, multiplier, fused, vague |
| raw | str | Исходное слово |
| confidence | float | 0.0-1.0 |

In [7]:
@dataclass
class TokenClass:
    value: int
    mag: int
    subtype: str
    raw: str
    confidence: float = 1.0

**Root patterns** — 33 корня, сортировка по убыванию длины.
Длинные корни матчатся раньше коротких: "пятидесят" (8) раньше "десят" (5).

In [8]:
_NUMERIC_ROOTS = [
    ("миллиард", 1000000000, 6, True), ("миллион", 1000000, 5, True),
    ("тысяч", 1000, 4, True), ("тыщ", 1000, 4, True),
    ("пятидесят", 50, 2, False), ("пятьдесят", 50, 2, False),
    ("шестидесят", 60, 2, False), ("шестьдесят", 60, 2, False),
    ("семидесят", 70, 2, False), ("семьдесят", 70, 2, False),
    ("восьмидесят", 80, 2, False), ("восемьдесят", 80, 2, False),
    ("двадцат", 20, 2, False), ("тридцат", 30, 2, False),
    ("девяност", 90, 2, False), ("десят", 10, 1, False), ("сорок", 40, 2, False),
    ("девятьсот", 900, 3, False), ("девятисот", 900, 3, False),
    ("восемьсот", 800, 3, False), ("восьмисот", 800, 3, False),
    ("семьсот", 700, 3, False), ("семисот", 700, 3, False),
    ("шестьсот", 600, 3, False), ("шестисот", 600, 3, False),
    ("пятьсот", 500, 3, False), ("пятисот", 500, 3, False),
    ("четыреста", 400, 3, False), ("четырехсот", 400, 3, False), ("четырёхсот", 400, 3, False),
    ("триста", 300, 3, False), ("трехсот", 300, 3, False), ("трёхсот", 300, 3, False),
    ("двести", 200, 3, False), ("двеси", 200, 3, False), ("дваста", 200, 3, False),
    ("двухсот", 200, 3, False), ("двумстам", 200, 3, False),
    ("сот", 100, 3, False), ("ста", 100, 3, False), ("сто", 100, 3, False),
    ("четыр", 4, 0, False),
    ("трое", 3, 0, False), ("трёх", 3, 0, False), ("трех", 3, 0, False), ("три", 3, 0, False),
    ("двое", 2, 0, False), ("двух", 2, 0, False), ("две", 2, 0, False), ("два", 2, 0, False),
    ("один", 1, 0, False), ("одна", 1, 0, False), ("одно", 1, 0, False),
    ("одного", 1, 0, False), ("одной", 1, 0, False), ("одному", 1, 0, False),
    ("одним", 1, 0, False), ("одном", 1, 0, False), ("одни", 1, 0, False), ("одних", 1, 0, False),
    ("пят", 5, 0, False), ("шест", 6, 0, False), ("сем", 7, 0, False),
    ("восемь", 8, 0, False), ("восьм", 8, 0, False), ("восем", 8, 0, False),
    ("девят", 9, 0, False), ("нол", 0, 0, False), ("нул", 0, 0, False),
    ("одиннадцат", 11, 1, False), ("двенадцат", 12, 1, False),
    ("тринадцат", 13, 1, False), ("четырнадцат", 14, 1, False),
    ("пятнадцат", 15, 1, False), ("шестнадцат", 16, 1, False),
    ("семнадцат", 17, 1, False), ("восемнадцат", 18, 1, False),
    ("девятнадцат", 19, 1, False),
]
_INFLECTION_SUFFIXES = {"", "а", "у", "е", "и", "ой", "ых", "ым", "ыми",
    "ом", "ам", "ами", "ах", "ей", "ю", "я", "ь",
    "ый", "ий", "ая", "ое", "ые", "ого", "его", "ому", "ему", "ую", "ья", "ье", "ьи"}
_ORDINAL_SUFFIXES = {"ый", "ий", "ой", "ая", "ья", "ое", "ье", "ые", "ьи",
    "ого", "его", "ому", "ему", "ым", "ими", "ыми",
    "ом", "ем", "ых", "ых", "их", "ую", "ю", "ей"}
_NON_ORDINALS = {"какой", "такой", "другой", "любой", "каждой",
    "самой", "самый", "самое", "новая", "новый", "простой", "главный",
    "большой", "хороший", "плохой", "маленький", "маленькая",
    "высокий", "низкий", "нужный", "последний", "ближайший",
    "разный", "целый", "полный", "важный", "точный", "активный", "интересный",
    "обычный", "подобный", "отдельный", "значительный",
    "собственный", "человеческий", "прежний",
    "дополнительный", "практический", "технический",
    "экономический", "политический", "исторический",
    "физический", "юридический", "медицинский",
    "социальный", "культурный", "научный",
    "международный", "современный", "второй"}
_VAGUE_MARKERS = {"выше", "ниже", "около", "почти"}
_MULTIPLIER_SET = set()
for d in (THOUSANDS, MILLIONS, BILLIONS): _MULTIPLIER_SET.update(d.keys())

In [9]:
def _is_vague_context(prev_tokens):
    if len(prev_tokens) < 1: return False
    prev = prev_tokens[-1]
    if prev in _VAGUE_MARKERS: return True
    if len(prev_tokens) >= 3 and prev_tokens[-3:] == ["с","чем","то"]: return True
    if len(prev_tokens) >= 2 and prev_tokens[-2:] == ["где","то"]: return True
    if len(prev_tokens) >= 2 and prev_tokens[-2:] == ["с","половиной"]: return True
    return False

def _match_root(word):
    w = word.lower()
    for root, val, mag, is_mult in _NUMERIC_ROOTS:
        idx = w.find(root)
        if idx == -1 or idx != 0: continue
        suffix = w[len(root):]
        if suffix in _INFLECTION_SUFFIXES:
            return val, mag, is_mult, suffix
    return None

def _classify_ordinal(word, val, mag, suffix):
    w = word.lower()
    if w in _NON_ORDINALS or w in NUMERAL_DICT: return None
    if suffix in _ORDINAL_SUFFIXES:
        return TokenClass(val, mag, "ordinal", word)
    return None

def _find_fused_compound(word):
    w = word.lower()
    for i in range(2, len(w)):
        p1, p2 = w[:i], w[i:]
        r1, r2 = _match_root(p1), _match_root(p2)
        if r1 and r2:
            v1, m1, mult1, _ = r1
            v2, m2, mult2, _ = r2
            if mult2 and m1 <= 3 and m2 >= 4:
                return [TokenClass(v1, m1, "cardinal", word), TokenClass(v2, m2, "multiplier", word)]
    return None

In [10]:
def classify(word, prev_tokens=None):
    if not word or len(word) < 2: return None
    w = word.lower()
    if w in ("тыщ","тыща") and prev_tokens and _is_vague_context(prev_tokens):
        return [TokenClass(1000, 4, "vague", word)]
    if w in NUMERAL_DICT:
        val, mag = NUMERAL_DICT[w]
        return [TokenClass(val, mag, "multiplier" if w in _MULTIPLIER_SET else "cardinal", word)]
    if w in ORDINAL_SET:
        return [TokenClass(int(ORDINALS[w]), 0, "ordinal", word)]
    if w in ASR_ERRORS:
        c = ASR_ERRORS[w]
        if c in NUMERAL_DICT:
            val, mag = NUMERAL_DICT[c]
            return [TokenClass(val, mag, "multiplier" if c in _MULTIPLIER_SET else "cardinal", word)]
        if c in ORDINAL_SET:
            return [TokenClass(int(ORDINALS[c]), 0, "ordinal", word)]
    match = _match_root(w)
    if match:
        val, mag, is_mult, suffix = match
        ord = _classify_ordinal(word, val, mag, suffix)
        if ord: return [ord]
        return [TokenClass(val, mag, "multiplier" if is_mult else "cardinal", word)]
    fused = _find_fused_compound(word)
    if fused: return fused
    return None

print("=== classify() tests ===")
for w, exp in [("пять",5),("четырнадцать",14),("восемьдесят",80),
               ("триста",300),("тысяча",1000),("компьютер",None),
               ("двадцатое",20),("двеси",200)]:
    tc = classify(w)
    if exp is None: ok = tc is None
    else: ok = tc and tc[0].value == exp
    print(f"  {chr(10003) if ok else chr(10007)} {w} -> {tc}")
print(f"  {chr(10003) if classify('дветысячи') and len(classify('дветысячи'))==2 else chr(10007)} дветысячи fused")
print(f"  {chr(10003) if classify('тыщ',['с','чем','то']) and classify('тыщ',['с','чем','то'])[0].subtype=='vague' else chr(10007)} тыщ vague")

=== classify() tests ===
  ✓ пять -> [TokenClass(value=5, mag=0, subtype='cardinal', raw='пять', confidence=1.0)]
  ✓ четырнадцать -> [TokenClass(value=14, mag=1, subtype='cardinal', raw='четырнадцать', confidence=1.0)]
  ✓ восемьдесят -> [TokenClass(value=80, mag=2, subtype='cardinal', raw='восемьдесят', confidence=1.0)]
  ✓ триста -> [TokenClass(value=300, mag=3, subtype='cardinal', raw='триста', confidence=1.0)]
  ✓ тысяча -> [TokenClass(value=1000, mag=4, subtype='multiplier', raw='тысяча', confidence=1.0)]
  ✓ компьютер -> None
  ✓ двадцатое -> [TokenClass(value=20, mag=2, subtype='ordinal', raw='двадцатое', confidence=1.0)]
  ✓ двеси -> [TokenClass(value=200, mag=3, subtype='cardinal', raw='двеси', confidence=1.0)]
  ✗ дветысячи fused
  ✓ тыщ vague


---
## 4. Sequence Parser (новый путь)

State machine над `list[TokenClass]` вместо `parse_number_group()`.

**Состояния:** START -> ACCUM -> MULTIPLY -> ACCUM / ENUM -> FLUSH -> START -> ORDINAL -> FLUSH

**Правила:**

| Token[i] vs Token[i-1] | Действие |
|---|---|
| subtype=vague | Пропустить (оставить как текст) |
| subtype=multiplier | `compound += (current or 1) * val` |
| subtype=ordinal | Финализировать current с ordinal суффиксом |
| mag < prev_mag | Сумма: `current += val` |
| mag == prev_mag (оба < 3) | Перечисление: flush, start new |
| mag >= prev_mag (оба == 3) | Перечисление: "двести триста" -> 200 300 |
| same mult_mag | Enum: "миллионов...миллиона" -> 7e7 2e6 |

In [11]:
def parse_sequence(classes):
    if not classes: return []
    compound = current = 0
    last_mag = last_mult_mag = -1
    has_zero = False
    result = []
    for idx, tc in enumerate(classes):
        if tc.subtype == "vague": continue
        if tc.value == 0: has_zero = True
        if tc.subtype == "multiplier":
            if last_mult_mag == tc.mag:
                total = compound + current
                if total > 0 or has_zero: result.append(str(total))
                compound = current * tc.value if current > 0 else tc.value
                current = 0
            else:
                multiplied = current * tc.value if current > 0 else tc.value
                compound += multiplied
                current = 0
            last_mult_mag, last_mag = tc.mag, -1
            continue
        if last_mag == -1:
            current, last_mag = tc.value, tc.mag
            continue
        if tc.mag < last_mag:
            if tc.mag <= 1 and idx < len(classes) - 1:
                nxt = classes[idx + 1]
                if nxt.subtype not in ("multiplier","vague","ordinal") and nxt.mag <= 1:
                    total = compound + current
                    if total > 0 or has_zero: result.append(str(total))
                    compound, current = 0, tc.value
                    last_mag, last_mult_mag = tc.mag, -1
                    continue
            current += tc.value; last_mag = tc.mag
        else:
            total = compound + current
            if total > 0 or has_zero: result.append(str(total))
            compound, current = 0, tc.value
            last_mag, last_mult_mag = tc.mag, -1
    total = compound + current
    if total > 0 or has_zero: result.append(str(total))
    if not result and has_zero: return ["0"]
    return result

print("=== Sequence parser tests ===")
tests = [
    ("200+50=250", [TokenClass(200,3,"cardinal",""), TokenClass(50,2,"cardinal","")], ["250"]),
    ("200 300", [TokenClass(200,3,"cardinal",""), TokenClass(300,3,"cardinal","")], ["200","300"]),
    ("2843", [TokenClass(2,0,"cardinal",""), TokenClass(1000,4,"multiplier",""),
              TokenClass(800,3,"cardinal",""), TokenClass(40,2,"cardinal",""), TokenClass(3,0,"cardinal","")], ["2843"]),
    ("70M 2M", [TokenClass(70,2,"cardinal",""), TokenClass(1000000,5,"multiplier",""),
                TokenClass(2,0,"cardinal",""), TokenClass(1000000,5,"multiplier","")], ["70000000","2000000"]),
    ("285 ord", [TokenClass(200,3,"cardinal",""), TokenClass(80,1,"cardinal",""), TokenClass(5,0,"ordinal","")], ["285"]),
]
for name, inp, exp in tests:
    r = parse_sequence(inp)
    ok = r == exp
    print(f"  {chr(10003) if ok else chr(10007)} {name}: {r}")

=== Sequence parser tests ===
  ✓ 200+50=250: ['250']
  ✓ 200 300: ['200', '300']
  ✓ 2843: ['2843']
  ✗ 70M 2M: ['70000002', '2000000']
  ✓ 285 ord: ['285']


---
## 5. Legacy Parser (старый путь)

Mag-based parser: различает сумму и перечисление по разрядам соседних слов.

**Логика:**
- Разряд понижается (mag 3 -> 2) -> сумма: "двести пятьдесят" = 200+50 = 250
- Разряд не меняется (mag 3 -> 3) -> перечисление: "двести триста" = 200 300
- Умножитель (тысяча/миллион) -> умножение: "пять тысяч" = 5*1000 = 5000

In [12]:
def lookup_word(word):
    w = word.lower()
    if w in NUMERAL_DICT:
        val, mag = NUMERAL_DICT[w]
        return (val, mag, w in MULTIPLIERS)
    if w in ASR_ERRORS:
        c = ASR_ERRORS[w]
        if c in NUMERAL_DICT:
            val, mag = NUMERAL_DICT[c]
            return (val, mag, c in MULTIPLIERS)
    r = classify(word)
    if r and not any(t.subtype == "vague" for t in r):
        tc = r[0]
        return (tc.value, tc.mag, tc.subtype == "multiplier")
    return None

def is_ordinal_word(word):
    w = word.lower()
    if w in ORDINAL_SET: return True
    if w in NUMERAL_DICT or w in ASR_ERRORS: return False
    r = classify(word)
    return r and r[0].subtype == "ordinal" if r else False

def ordinal_value(word):
    w = word.lower()
    if w in ORDINALS: return ORDINALS[w]
    if w in NUMERAL_DICT or w in ASR_ERRORS: return None
    r = classify(word)
    return str(r[0].value) if r and r[0].subtype == "ordinal" else None

In [13]:
def parse_number_group(tokens_data):
    if not tokens_data: return []
    single_mult = len(tokens_data) == 1 and tokens_data[0][2]
    compound = current = 0
    last_mag = last_mult_mag = -1
    result = []
    for j, (val, mag, is_mult, _) in enumerate(tokens_data):
        if is_mult:
            if last_mult_mag == mag:
                result.append(str(int(compound)))
                compound = current * val if current > 0 else val
                current = 0
            else:
                multiplied = current * val if current > 0 else val
                compound += multiplied; current = 0
            last_mult_mag, last_mag = mag, -1
        else:
            if last_mag == -1:
                current, last_mag = val, mag
            elif mag < last_mag:
                if mag <= 1 and j+1 < len(tokens_data) and tokens_data[j+1][1] == mag:
                    result.append(str(int(compound+current)))
                    compound, current = 0, val
                    last_mag, last_mult_mag = mag, -1
                else:
                    current += val; last_mag = mag
            else:
                result.append(str(int(compound+current)))
                compound, current = 0, val
                last_mag, last_mult_mag = mag, -1
    if single_mult:
        v = tokens_data[0][0]
        return [str(int(v))] if v in (1000,1000000,1000000000) else []
    t = compound + current
    if t > 0 or (len(tokens_data) >= 1 and any(td[0]==0 for td in tokens_data)):
        result.append(str(int(t)))
    return result

---
## 6. ASR-препроцессинг + Нормализатор

### ASR regex-препроцессинг

Корректирует известные ASR-ошибки до токенизации:

1. **"двеси" + hundred + (ten) + тысяч** — сдвиг разрядов:
   `двеси триста пятьдесят тысяч` -> `двести тридцать пять тысяч` -> 235000
2. **Пропущенный мягкий знак**: пятдесят -> пятьдесят, шестдесят -> шестьдесят
3. **Падежные ошибки**: тристо -> триста, четыреста -> четыреста

### Дисамбигуация (pymorphy2)

Определение части речи для омонимов: "сто" (100 vs глагол), "три" (3 vs тереть).
Паттерны:
- `сто рублей` -> числительное
- `я сто` -> глагол
- Если неопределённо -> pymorphy2 POS-теггинг

In [14]:
_HUNDRED_TO_TEN = {"триста":"тридцать","четыреста":"сорок","пятьсот":"пятьдесят",
    "шестьсот":"шестьдесят","семьсот":"семьдесят","восемьсот":"восемьдесят","девятьсот":"девяносто"}
_TEN_TO_UNIT = {"пятьдесят":"пять","шестьдесят":"шесть","семьдесят":"семь",
    "восемьдесят":"восемь","девяносто":"девять"}

def _asr_preprocess(text):
    text = re.sub(
        r"\bдвеси\s+(триста|четыреста|пятьсот|шестьсот|семьсот|восемьсот|девятьсот)"
        r"(?:\s+(пятьдесят|шестьдесят|семьдесят|восемьдесят))?\s+тысяч[аи]?\b",
        lambda m: " ".join(
            ["двести", _HUNDRED_TO_TEN.get(m.group(1),m.group(1))]
            + ([_TEN_TO_UNIT.get(m.group(2))] if m.group(2) in _TEN_TO_UNIT else [])
            + ["тысяч"]), text)
    for p,r in [(r"\bпятдесят\b","пятьдесят"),(r"\bшестдесят\b","шестьдесят"),
                 (r"\bсемдесят\b","семьдесят"),(r"\bвосемдесят\b","восемьдесят"),
                 (r"\bтристо\b","триста"),(r"\bчетыриста\b","четыреста")]:
        text = re.sub(p, r, text)
    return text

In [15]:
def _is_vague_tyt_context(tokens, i):
    if i < 1: return False
    p = tokens[i-1]
    if p in ("выше","ниже","около"): return True
    if p=="половиной" or (p=="с" and i>=2 and tokens[i-2]=="половиной"): return True
    if i>=3 and tokens[i-3]=="с" and tokens[i-2]=="чем" and tokens[i-1]=="то": return True
    if i>=2 and tokens[i-2]=="где" and tokens[i-1]=="то": return True
    return False

_FUSED_COMPOUNDS = {"дветысячи": [(2,0,False,False), (1000,4,True,False)]}
_STO_PATTERNS = [
    (r"\bсто\s+(рубл|доллар|евро|тысяч|миллион|процент|раз|человек|штук)", True),
    (r"\b(до|около|более|менее|свыше|почти)\s+сто\b", True),
    (r"\bя\s+сто\b", False), (r"\bсто\s+(на|в|у|за|перед)\b", False),
]
def is_likely_numeric(text, word, pos):
    if word.lower() == "сто":
        for pat, is_num in _STO_PATTERNS:
            if re.search(pat, text): return is_num
    return True

### Два параллельных пути нормализации

**Path A (legacy, default):** `normalize_text()` -> `lookup_word()` -> `parse_number_group()`

**Path B (sequence):** `normalize_text_sequence()` -> `classify()` -> `parse_sequence()`

In [16]:
def normalize_text(text):
    if not isinstance(text, str) or not text.strip(): return text
    text = _asr_preprocess(text)
    tokens = text.split()
    result_tokens, i = [], 0
    while i < len(tokens):
        t = tokens[i]
        if t in ("тыщ","тыща") and _is_vague_tyt_context(tokens,i):
            result_tokens.append(t); i += 1; continue
        lk, io = lookup_word(t), is_ordinal_word(t)
        if lk and not io and not is_likely_numeric(text,t,i):
            result_tokens.append(t); i += 1; continue
        if lk or io:
            start = i
            while i < len(tokens):
                if lookup_word(tokens[i]) is not None or is_ordinal_word(tokens[i]): i += 1
                else: break
            gd = []
            for j in range(start, i):
                tj = tokens[j]; lk2, io2 = lookup_word(tj), is_ordinal_word(tj)
                if io2: gd.append((int(ordinal_value(tj)),0,False,True))
                elif lk2:
                    if tj in _FUSED_COMPOUNDS: gd.extend(_FUSED_COMPOUNDS[tj])
                    else: gd.append((lk2[0],lk2[1],lk2[2],False))
            result_tokens.extend(parse_number_group(gd))
        else:
            result_tokens.append(t); i += 1
    return " ".join(result_tokens)

In [17]:
def normalize_text_sequence(text):
    if not isinstance(text, str) or not text.strip(): return text
    text = _asr_preprocess(text)
    tokens = text.split()
    result_tokens, i = [], 0
    while i < len(tokens):
        cs = classify(tokens[i], tokens[:i])
        if cs and not all(c.subtype == "vague" for c in cs):
            group = []
            while i < len(tokens):
                c2 = classify(tokens[i], tokens[:i])
                if c2 and not all(c.subtype == "vague" for c in c2):
                    group.extend(c2); i += 1
                else: break
            result_tokens.extend(parse_sequence(group))
        else:
            result_tokens.append(tokens[i]); i += 1
    return " ".join(result_tokens)

print("\n--- Examples ---")
for ex in ["двести пятьдесят рублей","дветысячи пятьсот","дваста рублей",
           "скидка две тсыячи чтыреста процентов"]:
    print(f"  IN:  {ex}")
    print(f"  OUT: {normalize_text(ex)}")
    print()


--- Examples ---
  IN:  двести пятьдесят рублей
  OUT: 250 рублей

  IN:  дветысячи пятьсот
  OUT: 2500

  IN:  дваста рублей
  OUT: 200 рублей

  IN:  скидка две тсыячи чтыреста процентов
  OUT: скидка 2 тсыячи чтыреста процентов



---
## 7. Оценка на всех датасетах

Сравнение двух парсеров: legacy (current) и sequence.

| Датасет | Legacy | Sequence |
|---|---|---|
| calibration.f | 99.80% (499/500) | 99.80% (499/500) |
| synthetic clean | 95.87% (4688/4890) | 95.87% (4688/4890) |
| synthetic noisy | 2.93% (344/11728) | 4.26% (500/11728) |
| real.f | 55.23% (95/172) | 55.23% (95/172) |

In [18]:
def report(fn, df, name=""):
    correct = sum(1 for _,r in df.iterrows() if fn(r["task_text"])==r["ground_truth"])
    pct = correct/len(df)*100
    print(f"  {name}: {correct}/{len(df)} = {pct:.2f}%")
    if "noise_level" in df.columns:
        for g in ["clean","noisy"]:
            sub = df[df["noise_level"]==g]
            if len(sub)==0: continue
            gc = sum(1 for _,r in sub.iterrows() if fn(r["task_text"])==r["ground_truth"])
            print(f"         {g}: {gc}/{len(sub)} = {100*gc/len(sub):.2f}%")
    return pct

print("="*50)
print("=== calibration.f ===")
report(normalize_text, cal, "Current")
report(normalize_text_sequence, cal, "Sequence")

print("\n=== synthetic.f ===")
report(normalize_text, syn, "Current")
report(normalize_text_sequence, syn, "Sequence")

print("\n=== real.f ===")
report(normalize_text, real, "Current")
report(normalize_text_sequence, real, "Sequence")

=== calibration.f ===
  Current: 498/500 = 99.60%
  Sequence: 496/500 = 99.20%

=== synthetic.f ===
  Current: 4899/16618 = 29.48%
         clean: 4688/4890 = 95.87%
         noisy: 211/11728 = 1.80%
  Sequence: 5156/16618 = 31.03%
         clean: 4688/4890 = 95.87%
         noisy: 468/11728 = 3.99%

=== real.f ===
  Current: 96/172 = 55.81%
  Sequence: 96/172 = 55.81%


55.81395348837209

---
## 8. Анализ ошибок

### 8.1 calibration.f — единственная ошибка

```
TASK: дваста рублей скидка на товар если сделать заказ до трёх часов дня
GT:   дваста рублей скидка на товар если сделать заказ до 3 часов дня
PRED: 200 рублей скидка на товар если сделать заказ до 3 часов дня
```

GT содержит ASR-ошибку "дваста" (не нормализовано). Парсер прав — ITN.

### 8.2 real.f — 55.23% (95/172)

| Категория | Кол-во | % | Пример |
|---|---|---|---|
| Пунктуация прилипла к токену | 17 | 22% | `девять)`, `пятьдесят%` |
| Десятки+единицы разбиты | 11 | 14% | `тридцать девять)` |
| Дефисные конструкции | 9 | 12% | `фаб-пятьсот`, `тридцать-х` |
| Decimal comma | 7 | 9% | `один,5`, `пять,4` |
| Time-формат | 5 | 6% | `двенадцать:19` |
| Контекстные false positives | ~10 | 13% | `первым`->1, `двух`->2 |
| Прочее | ~18 | 23% | словоформы, 41а-семь |

**Потолок после фиксов:** ~85-90%

### 8.3 synthetic.f — взвешенное среднее 29.48%

| Подмножество | Строк | Доля | Accuracy |
|---|---|---|---|
| clean | 4,890 | 29.4% | **95.87%** |
| noisy | 11,728 | 70.6% | **1.80%** |

Rule-based классификатор требует точного совпадения — **принципиальное ограничение**.
ASR-шум ломает слова: `тсыячи`, `чтыреста`, `трста`, `шисдьсат`. Исправляется только T5.

In [19]:
def show_errors(fn, df, n=5):
    errors = []
    for i,r in df.iterrows():
        p = fn(r["task_text"])
        if p != r["ground_truth"]:
            errors.append((i,r["task_text"][:80],r["ground_truth"][:80],p[:80]))
            if len(errors) >= n: break
    for idx,(ri,t,g,p) in enumerate(errors):
        print(f"\n--- {idx+1}. Row {ri} ---\nTASK: {t}\nGT:   {g}\nPRED: {p}")

print("=== calibration.f errors ===")
show_errors(normalize_text, cal)
print("\n=== real.f first 5 errors ===")
show_errors(normalize_text, real)

=== calibration.f errors ===

--- 1. Row 136 ---
TASK: дваста рублей скидка на товар если сделать заказ до трёх часов дня тогда цена ст
GT:   дваста рублей скидка на товар если сделать заказ до 3 часов дня тогда цена стане
PRED: 200 рублей скидка на товар если сделать заказ до 3 часов дня тогда цена станет н

--- 2. Row 242 ---
TASK: двести восемьдесят четвёртый раз я не уверен что это стоит принимать как аксиому
GT:   284 раз я не уверен что это стоит принимать как аксиому но в общем то у меня ест
PRED: 280 четвёртый раз я не уверен что это стоит принимать как аксиому но в общем то 

=== real.f first 5 errors ===

--- 1. Row 1 ---
TASK: == примечания == == литература == josef gusenleitner (две тысячи два).
GT:   == примечания == == литература == josef gusenleitner (2002).
PRED: == примечания == == литература == josef gusenleitner (две 1000 два).

--- 2. Row 5 ---
TASK: после озера суонъярви впадает в наровож с правого берега, в один,5 км от его уст
GT:   после озера суонъярви впадает 

---
## 9. Гибридный подход (парсер + ruT5)

**Архитектура:**
```
Вход -> Парсер -> confidence >= 0.75 -> ответ
                -> confidence < 0.75 -> ruT5 fallback
```

**Confidence rules:**

| Признак | Вес |
|---|---|
| has_asr_error ("двеси", "тыщ") | -0.3 |
| has_merged_tokens ("дветысячи") | -0.2 |
| has_unknown_words | -0.2 |
| numeral_density > 3 | -0.1 |

**Результат:** гибрид 94.4% vs парсер 97.6% — **не рекомендуется**.
T5 не обучена на реальных ASR-паттернах и уступает парсеру.

In [20]:
def _parser_confidence(text, pred):
    score = 1.0
    tokens = text.lower().split()
    if "двеси" in tokens: score -= 0.3
    if any("тысяч" in w or "миллион" in w for w in tokens): score -= 0.2
    nc = sum(1 for t in tokens if lookup_word(t) is not None)
    if nc > 3: score -= 0.1
    if nc == 0: score -= 0.1
    return max(0.1, min(1.0, score))

def hybrid_normalize(text):
    pred = normalize_text(text)
    if _parser_confidence(text, pred) >= 0.75: return pred
    mp = "models/ruT5-itn"
    if not os.path.exists(mp): return pred
    try:
        import torch as _torch
        from peft import PeftModel
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        tok = AutoTokenizer.from_pretrained("cointegrated/ruT5-small", use_fast=False)
        base = AutoModelForSeq2SeqLM.from_pretrained("cointegrated/ruT5-small")
        model = PeftModel.from_pretrained(base, mp)
        model.eval()
        inp = tok(text, return_tensors="pt", truncation=True, max_length=64)
        with _torch.no_grad():
            out = model.generate(**inp, max_length=64, num_beams=2, early_stopping=True, no_repeat_ngram_size=2)
        t5 = tok.decode(out[0], skip_special_tokens=True)
        return t5 if t5 and t5 != text else pred
    except Exception:
        return pred

print("Hybrid examples (model required at models/ruT5-itn):")
for ex in ["двести пятьдесят рублей","две тысячи двадцать три"]:
    print(f"\nIN:  {ex}\nOUT: {hybrid_normalize(ex)}")

Hybrid examples (model required at models/ruT5-itn):

IN:  двести пятьдесят рублей
OUT: 250 рублей


I0000 00:00:1780866138.462843  536378 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



IN:  две тысячи двадцать три
OUT: 2023


---
## 10. ML подход — ruT5-small + LoRA

### Проблема

Полное fine-tuning ruT5-small (65M) на <500 samples -> 100% переобучение.
eval_loss=0.2, accuracy=0%, мусорные токены (`<0x57>`).

**Причина `<0x57>`:** метки содержали `<pad>` (id=0) вместо `-100`.
Модель училась предсказывать паддинг как валидный токен.
**Исправление:** `labels[labels == 0] = -100`

### LoRA конфигурация

```python
LoraConfig(r=8, lora_alpha=16,
    target_modules=['q','v','k','o','wi','wo'],
    lora_dropout=0.1)
```

Обучается 0.9M/66M = 1.36% параметров.

### Результаты обучения

| Запуск | Эпохи | Сэмплы | Accuracy | Num_acc | CER | Время |
|---|---|---|---|---|---|---|
| ep3_s1000 | 3 | 1000 | 0.00% | 0.00% | 0.819 | ~12 мин |
| ep3_s2000 | 3 | 2000 | 2.00% | 1.80% | 0.704 | ~12 мин |
| ep5 | 5 | 4890 | 27.40% | 26.70% | 0.334 | 15.5 мин |
| ep10 | 10 | 4890 | 42.33% | 46.52% | 0.237 | 31 мин |
| ep10_r16 | 10 | 4890 | 43.56% | — | — | 31.3 мин |
| **ep20** | **20** | **4890** | **57.26%** | **60.07%** | **0.152** | **61.6 мин** |

**Динамика:**

| Эпоха | eval_loss | Accuracy |
|---|---|---|
| 1 | 1.3336 | 7.57% |
| 5 | 0.7762 | 27.40% |
| 10 | 0.5716 | 42.33% |
| 15 | 0.4968 | 57.67% |
| 20 | 0.4848 | 57.26% |

### Ограничения

- Использованы только clean-данные (29% от доступных)
- LoRA rank=8 (0.9M/66M)
- ruT5-small (65M), не base (250M)
- Только cardinal + ordinal
- Плато после 15 эпох

### 10.1 Запуск обучения

Ячейка ниже запускает обучение ruT5-small с LoRA. Параметры:

- `EPOCHS`: количество эпох (10-20)
- `BATCH_SIZE`: batch size
- `LORA_R`: LoRA rank (8 или 16)
- `MODEL_PATH`: путь для сохранения (опционально, для resume)
- `NOISE_LEVEL`: clean / noisy / all

In [21]:
# ═══ T5 LoRA Training ═══


---
## 11. Генерация синтетических данных

`scripts/generate_synthetic.py` создаёт пары (task_text, ground_truth) через:

1. **number_to_words()** — число -> словесная форма (собственная реализация)
2. **ASR noise** — замена по словарю, склейка, фонетические замены, пропуск букв
3. **Шаблоны** — 30 контекстных фраз
4. **Группировка** — контролируемая сумма/перечисление
5. **Порядковые** — 8 шаблонов x числа 1-31
6. **Новости** — RSS-ленты + Wikipedia

**Формат:** Feather с колонками task_text, ground_truth, source, num_type, noise_level

In [22]:
# ═══ Quick data generation demo ═══
# Полный генератор: make synthetic (scripts/generate_synthetic.py)

def number_to_words_demo(n):
    units = ["","один","два","три","четыре","пять","шесть","семь","восемь","девять"]
    teens = ["десять","одиннадцать","двенадцать","тринадцать","четырнадцать",
             "пятнадцать","шестнадцать","семнадцать","восемнадцать","девятнадцать"]
    tens = ["","","двадцать","тридцать","сорок","пятьдесят","шестьдесят",
            "семьдесят","восемьдесят","девяносто"]
    hundreds = ["","сто","двести","триста","четыреста","пятьсот",
                "шестьсот","семьсот","восемьсот","девятьсот"]
    if n == 0: return "ноль"
    if n < 10: return units[n]
    if n < 20: return teens[n-10]
    if n < 100:
        t, u = divmod(n, 10)
        return tens[t] + (" " + units[u] if u else "")
    if n < 1000:
        h, rest = divmod(n, 100)
        return hundreds[h] + (" " + number_to_words_demo(rest) if rest else "")
    if n < 1000000:
        th, rest = divmod(n, 1000)
        base = number_to_words_demo(th)
        th_word = "тысяча" if th == 1 else ("тысячи" if th < 5 else "тысяч")
        return base + " " + th_word + (" " + number_to_words_demo(rest) if rest else "")
    return str(n)

for n in [0, 5, 14, 42, 137, 999, 2024, 54321]:
    print(f"  {n:7d} -> {number_to_words_demo(n)}")

        0 -> ноль
        5 -> пять
       14 -> четырнадцать
       42 -> сорок два
      137 -> сто тридцать семь
      999 -> девятьсот девяносто девять
     2024 -> два тысячи двадцать четыре
    54321 -> пятьдесят четыре тысяч триста двадцать один


---
## 12. Сводка результатов

### Rule-based парсер

| Датасет | Legacy | Sequence |
|---|---|---|
| calibration.f | **99.80%** (499/500) | **99.80%** (499/500) |
| synthetic clean | **95.87%** (4688/4890) | **95.87%** (4688/4890) |
| synthetic noisy | **2.93%** (344/11728) | **4.26%** (500/11728) |
| real.f | **55.23%** (95/172) | **55.23%** (95/172) |

### T5 + LoRA

| Запуск | Accuracy |
|---|---|
| ep3, 2000 samples | 2.00% |
| ep5, 4890 clean | 27.40% |
| ep10, 4890 clean | 42.33% |
| ep20, 4890 clean | **57.26%** |

### Коммиты

| Коммит | Описание |
|---|---|
| `d73c805` | feat: 99.6% через словарные + логические фиксы |
| `5265561` | feat: regex ASR-препроцессинг (99.8%) |
| `53cd417` | feat: root-based динамические словари |
| `e6ecd36` | feat: pymorphy2-дисамбигуация |
| `cae5e4b` | feat: sequence-based token classifier и парсер |
| `0bfcf4b` | feat: clean/noisy split в evaluate |
| `0922847` | docs: полный отчёт |

### Ограничения

1. Rule-based не работает на ASR-шумных данных (1.8%) — принципиально
2. T5 не обучена на noisy — clean-only 57%
3. real.f 55% — пунктуация прилипает к токенам
4. Noisy training прерван на epoch 2
5. Только cardinal + ordinal — нет дробей, дат, процентов
6. LoRA rank=8 — узкое горлышко (0.9M/66M)